In [5]:
pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 124.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 94.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 703.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 103.4 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalli

In [6]:
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [13]:
import shutil
import os
import glob

# List of paths to remove
paths_to_remove = [
    "/content/SDS-CP028",                        # Flattened folder
    "/content/kaggle",                           # Kaggle directory (if used)
    "/content/__MACOSX",                         # MacOS artifacts
    "/root/.kaggle/kaggle.json",                 # Default Kaggle key
    "/root/.kaggle/kaggle2.json",                # Alternate Kaggle key
    "/content/kaggle/input/new-bangladeshi-crop-disease",  # Raw input
    "/content/SmartLeaf_dataset",
    "/content/YOLO_dataset",
    "/content/yolo_dataset"
]

# Delete individual paths
for path in paths_to_remove:
    if os.path.exists(path):
        print(f"🧹 Deleting: {path}")
        if os.path.isdir(path):
            shutil.rmtree(path)
        else:
            os.remove(path)

# Delete any folder matching the Kaggle dataset pattern
for folder in glob.glob("/content/*new-bangladeshi-crop-disease*"):
    print(f"🧹 Removing leftover: {folder}")
    shutil.rmtree(folder, ignore_errors=True)

# Optional: Clean ~/.kaggle folder
if os.path.exists("/root/.kaggle"):
    print("🧹 Resetting ~/.kaggle directory")
    for file in os.listdir("/root/.kaggle"):
        if file.endswith(".json"):
            os.remove(f"/root/.kaggle/{file}")

print("✅ Cleanup complete.")



🧹 Deleting: /content/SDS-CP028
🧹 Deleting: /root/.kaggle/kaggle2.json
🧹 Deleting: /content/SmartLeaf_dataset
🧹 Resetting ~/.kaggle directory
✅ Cleanup complete.


In [14]:
from google.colab import drive
drive.mount("/content/drive")
from google.colab import files
files.upload()

!mkdir -p ~/.kaggle
!mv kaggle2.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle2.json



MessageError: Error: credential propagation was unsuccessful

In [15]:

import kagglehub

# Download latest version
path = kagglehub.dataset_download("nafishamoin/new-bangladeshi-crop-disease")

print("Path to dataset files:", path)




100%|██████████| 2.35G/2.35G [00:30<00:00, 82.5MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/nafishamoin/new-bangladeshi-crop-disease/versions/2


In [16]:
!mkdir -p ./SDS-CP028
!mv /root/.cache/kagglehub/datasets/nafishamoin/new-bangladeshi-crop-disease/versions/2 ./SDS-CP028/

#!cp -r /kaggle/input/new-bangladeshi-crop-disease ./SDS-CP028/

In [17]:
import os
import shutil
import random

# Set random seed for reproducibility
random.seed(42)

# Paths/content/SDS-CP028/2/BangladeshiCrops/BangladeshiCrops/Crop___Disease
original_dataset_dir = './SDS-CP028/2/BangladeshiCrops/BangladeshiCrops/Crop___Disease'  # Root path where Corn, Potato, etc. folders are
base_dir = 'SmartLeaf_dataset'                            # Where you want to create train/val/test folders
train_dir = os.path.join(base_dir, 'train')
val_dir = os.path.join(base_dir, 'val')
test_dir = os.path.join(base_dir, 'test')

# Split ratios
train_ratio = 0.8
val_ratio = 0.1
# test_ratio = 0.1 (implicitly the remaining)

# Create train, val, and test directories
for split_dir in [train_dir, val_dir, test_dir]:
    os.makedirs(split_dir, exist_ok=True)

# Traverse two levels: crop -> class
for crop_folder in os.listdir(original_dataset_dir):
    crop_path = os.path.join(original_dataset_dir, crop_folder)

    if os.path.isdir(crop_path):
        # Now go inside each disease class
        for class_folder in os.listdir(crop_path):
            class_path = os.path.join(crop_path, class_folder)

            if os.path.isdir(class_path):
                images = os.listdir(class_path)
                random.shuffle(images)

                total_images = len(images)
                train_split = int(total_images * train_ratio)
                val_split = int(total_images * (train_ratio + val_ratio))

                train_images = images[:train_split]
                val_images = images[train_split:val_split]
                test_images = images[val_split:]

                # Create corresponding class folders under train/val/test
                train_class_dir = os.path.join(train_dir, class_folder)
                val_class_dir = os.path.join(val_dir, class_folder)
                test_class_dir = os.path.join(test_dir, class_folder)

                os.makedirs(train_class_dir, exist_ok=True)
                os.makedirs(val_class_dir, exist_ok=True)
                os.makedirs(test_class_dir, exist_ok=True)

                # Copy images
                for img in train_images:
                    shutil.copy2(os.path.join(class_path, img), os.path.join(train_class_dir, img))

                for img in val_images:
                    shutil.copy2(os.path.join(class_path, img), os.path.join(val_class_dir, img))

                for img in test_images:
                    shutil.copy2(os.path.join(class_path, img), os.path.join(test_class_dir, img))

print("Dataset split into train/val/test successfully!")


Dataset split into train/val/test successfully!


In [18]:
from ultralytics import YOLO

# Load classification model (you can use yolov8n-cls, yolov8s-cls, yolov8m-cls, etc.)
model = YOLO('yolov8s-cls.pt')  # Small model for fast training

# Train
model.train(
    data='/content/SmartLeaf_dataset',
    epochs=20,
    imgsz=224,  # recommended input size for classification
    batch=32
)


Ultralytics 8.3.174 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/SmartLeaf_dataset, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=12.0, pretrain

AMP: checks passed ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 633.5±266.8 MB/s, size: 12.6 KB)


train: Scanning /content/SmartLeaf_dataset/train... 10412 images, 2 corrupt: 100%|██████████| 10414/10414 [00:02<00:00, 4884.74it/s]

train: /content/SmartLeaf_dataset/train/Wheat___Brown_Rust/Brown_rust203.jpg: ignoring corrupt image/label: image size (8, 48) <10 pixels
train: /content/SmartLeaf_dataset/train/Wheat___Yellow_Rust/Yellow_rust516.jpg: ignoring corrupt image/label: image size (1, 30) <10 pixels
train: New cache created: /content/SmartLeaf_dataset/train.cache


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 561.8±209.9 MB/s, size: 13.2 KB)


...

                   all      0.972          1



20 epochs completed in 1.101 hours.
Optimizer stripped from runs/classify/train/weights/last.pt, 10.3MB
Optimizer stripped from runs/classify/train/weights/best.pt, 10.3MB

Validating runs/classify/train/weights/best.pt...
Ultralytics 8.3.174 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
YOLOv8s-cls summary (fused): 30 layers, 5,093,134 parameters, 0 gradients, 12.5 GFLOPs
train: /content/SmartLeaf_dataset/train... found 10414 images in 14 classes ✅ 
val: /content/SmartLeaf_dataset/val... found 1301 images in 14 classes ✅ 
test: /content/SmartLeaf_dataset/test... found 1309 images in 14 classes ✅ 


               classes   top1_acc   top5_acc: 100%|██████████| 21/21 [00:17<00:00,  1.20it/s]


                   all      0.972          1
Speed: 0.1ms preprocess, 0.4ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to runs/classify/train


ultralytics.utils.metrics.ClassifyMetrics object with attributes:

confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7e8e584f8810>
curves: []
curves_results: []
fitness: 0.9861644804477692
keys: ['metrics/accuracy_top1', 'metrics/accuracy_top5']
results_dict: {'metrics/accuracy_top1': 0.9723289608955383, 'metrics/accuracy_top5': 1.0, 'fitness': 0.9861644804477692}
save_dir: PosixPath('runs/classify/train')
speed: {'preprocess': 0.07230313528035477, 'inference': 0.39441505841478125, 'loss': 0.0001960361266980176, 'postprocess': 0.0003407025350715273}
task: 'classify'
top1: 0.9723289608955383
top5: 1.0

YOLOv8 classification model has trained successfully and performed very well:

📊 Model Performance Summary
Metric	Value
Top-1 Accuracy	97.23%
Top-5 Accuracy	100%
Fitness Score	0.986
Inference Speed/Image	~0.40 ms
Preprocessing Time	~0.08 ms

This indicates the model is:

Accurately predicting the correct class as the top choice 97% of the time.

Always predicting the correct class among the top 5 choices.

Super fast and efficient, making it suitable for real-time classification on edge devices like Jetson Nano or in production environments.

In [19]:
#Validate

!yolo classify val model='runs/classify/train/weights/best.pt' data='SmartLeaf_dataset'

Ultralytics 8.3.174 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
YOLOv8s-cls summary (fused): 30 layers, 5,093,134 parameters, 0 gradients, 12.5 GFLOPs
train: /content/SmartLeaf_dataset/train... found 10414 images in 14 classes ✅ 
val: /content/SmartLeaf_dataset/val... found 1301 images in 14 classes ✅ 
test: /content/SmartLeaf_dataset/test... found 1309 images in 14 classes ✅ 
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 761.2±249.8 MB/s, size: 13.2 KB)
val: Scanning /content/SmartLeaf_dataset/val... 1301 images, 0 corrupt: 100% 1301/1301 [00:00<?, ?it/s]
               classes   top1_acc   top5_acc: 100% 82/82 [00:21<00:00,  3.88it/s]
                   all      0.972          1
Speed: 0.1ms preprocess, 1.0ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to runs/classify/val
💡 Learn more at https://docs.ultralytics.com/modes/val


In [20]:
from ultralytics import YOLO

model = YOLO('runs/classify/train/weights/best.pt')  # Load trained model
results = model.val(data='SmartLeaf_dataset')        # Run validation

print("Top-1 Accuracy:", results.results_dict['metrics/accuracy_top1'])
print("Top-5 Accuracy:", results.results_dict['metrics/accuracy_top5'])


Ultralytics 8.3.174 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
YOLOv8s-cls summary (fused): 30 layers, 5,093,134 parameters, 0 gradients, 12.5 GFLOPs
train: /content/SmartLeaf_dataset/train... found 10414 images in 14 classes ✅ 
val: /content/SmartLeaf_dataset/val... found 1301 images in 14 classes ✅ 
test: /content/SmartLeaf_dataset/test... found 1309 images in 14 classes ✅ 
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 624.4±236.6 MB/s, size: 13.2 KB)


val: Scanning /content/SmartLeaf_dataset/val... 1301 images, 0 corrupt: 100%|██████████| 1301/1301 [00:00<?, ?it/s]
               classes   top1_acc   top5_acc: 100%|██████████| 82/82 [00:19<00:00,  4.16it/s]


                   all      0.972          1
Speed: 0.1ms preprocess, 1.0ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to runs/classify/val2
Top-1 Accuracy: 0.9715603590011597
Top-5 Accuracy: 1.0


In [23]:
from ultralytics import YOLO

model = YOLO("runs/classify/train/weights/best.pt")
model.export(format="onnx")  # You can also use "torchscript", "openvino", etc.



Ultralytics 8.3.174 🚀 Python-3.11.13 torch-2.6.0+cu124 CPU (Intel Xeon 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLOv8s-cls summary (fused): 30 layers, 5,093,134 parameters, 0 gradients, 12.5 GFLOPs

PyTorch: starting from 'runs/classify/train/weights/best.pt' with input shape (1, 3, 224, 224) BCHW and output shape(s) (1, 14) (9.8 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<1.18.0', 'onnxslim>=0.1.59', 'onnxruntime-gpu'] not found, attempting AutoUpdate...

requirements: AutoUpdate success ✅ 7.8s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.17.0 opset 19...
ONNX: slimming with onnxslim 0.1.62...
ONNX: export success ✅ 8.7s, saved as 'runs/classify/train/weights/best.onnx' (19.5 MB)

Export complete (9.2s)
Results saved to /content/runs/classify/train/weights
Predict:         yolo predict 

'runs/classify/train/weights/best.onnx'

In [25]:
from ultralytics import YOLO

model = YOLO("runs/classify/train/weights/best.pt")
results = model.predict(source="/content/SmartLeaf_dataset/test/Corn___Northern_Leaf_Blight")  # Folder, image, video, etc.

# Optional: view results
for r in results:
    # Get the predicted class index and confidence
    top1_index = r.probs.top1
    top1_conf = r.probs.top1conf.item()

    # Get the class name using the model's class names
    predicted_class_name = model.names[top1_index]

    print(f"Predicted Class: {predicted_class_name}, Confidence: {top1_conf:.4f}")


image 1/99 /content/SmartLeaf_dataset/test/Corn___Northern_Leaf_Blight/image (108).JPG: 224x224 Corn___Northern_Leaf_Blight 1.00, Corn___Gray_Leaf_Spot 0.00, Corn___Common_Rust 0.00, Wheat___Brown_Rust 0.00, Rice___Brown_Spot 0.00, 4.6ms
image 2/99 /content/SmartLeaf_dataset/test/Corn___Northern_Leaf_Blight/image (133).JPG: 224x224 Corn___Northern_Leaf_Blight 1.00, Corn___Gray_Leaf_Spot 0.00, Rice___Brown_Spot 0.00, Rice___Leaf_Blast 0.00, Potato___Late_Blight 0.00, 3.0ms
image 3/99 /content/SmartLeaf_dataset/test/Corn___Northern_Leaf_Blight/image (134).JPG: 224x224 Corn___Northern_Leaf_Blight 1.00, Corn___Gray_Leaf_Spot 0.00, Rice___Brown_Spot 0.00, Potato___Late_Blight 0.00, Rice___Leaf_Blast 0.00, 3.0ms
image 4/99 /content/SmartLeaf_dataset/test/Corn___Northern_Leaf_Blight/image (137).JPG: 224x224 Corn___Northern_Leaf_Blight 1.00, Corn___Gray_Leaf_Spot 0.00, Corn___Common_Rust 0.00, Rice___Brown_Spot 0.00, Rice___Leaf_Blast 0.00, 3.0ms
image 5/99 /content/SmartLeaf_dataset/test/Cor

In [29]:
import gradio as gr
from PIL import Image, ImageDraw, ImageFont
from ultralytics import YOLO
import torch
import traceback

# Load YOLOv8 classification model
model = YOLO("runs/classify/train/weights/best.pt")

# Define prediction function
def classify_image(img):
    predictions_text = "Error processing image."
    annotated = img.copy() # Create a copy for drawing even if prediction fails

    try:
        results = model(img)
        if results:
            pred = results[0]
            probs = pred.probs  # class probabilities

            # Get top-3 class predictions
            topk = torch.topk(probs.data, k=3)
            class_ids = topk.indices.tolist()
            confidences = topk.values.tolist()

            # Get class names
            class_names = [model.names[i] for i in class_ids]

            # Format predictions for textbox
            predictions_text = "\n".join([f"{name}: {conf:.2f}" for name, conf in zip(class_names, confidences)])

            # Draw on image (Top-1 only)
            draw = ImageDraw.Draw(annotated)
            # Adjust font loading for broader compatibility
            try:
                font = ImageFont.truetype("arial.ttf", 20)
            except IOError:
                font = ImageFont.load_default()

            label_text = f"{class_names[0]} ({confidences[0]:.2f})"
            # Calculate text size and position
            # Using textbbox for more accurate bounding box calculation
            try:
                bbox = draw.textbbox((0, 0), label_text, font=font)
                text_width = bbox[2] - bbox[0]
                text_height = bbox[3] - bbox[1]
            except AttributeError:
                 # Fallback for older Pillow versions
                text_width, text_height = draw.textsize(label_text, font=font)


            # Draw background rectangle
            draw.rectangle([(0, 0), (text_width + 20, text_height + 10)], fill=(0, 0, 0))
            # Draw text
            draw.text((10, 5), label_text, fill=(255, 255, 255), font=font)
        else:
             predictions_text = "No results from model prediction."


    except Exception as e:
        predictions_text = f"An error occurred during classification: {e}\n{traceback.format_exc()}"
        print(predictions_text) # Print error to console as well


    return annotated, predictions_text

# Gradio interface
demo = gr.Interface(
    fn=classify_image,
    inputs=gr.Image(type="pil", label="Upload Image or Use Camera", sources=["upload", "webcam", "clipboard"]),
    outputs=[
        gr.Image(label="Prediction (Top-1 Overlay)"),
        gr.Textbox(label="Top 3 Predictions")
    ],
    title="YOLOv8 Leaf Disease Classifier",
    description="Upload an image or use your webcam to classify plant leaf diseases. Shows top-3 predictions with confidence.",
    allow_flagging="never"
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://60f04d3b70fb6e8ab4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
